In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.formula_1.silver_dim_drivers;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.formula_1.silver_dim_drivers (
  session_key INT,
  driver_number INT,
  name_acronym STRING,
  driver_name STRING,
  team_name STRING,
  team_hex_colour STRING
)
USING DELTA;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT driver_name) AS total_drivers,
    COUNT(DISTINCT team_name) AS total_teams 
FROM 
    workspace.formula_1.silver_dim_drivers

In [0]:
%sql
MERGE INTO workspace.formula_1.silver_dim_drivers AS target
USING (
  WITH parsed_drivers AS (
    SELECT
        TRY_CAST(get_json_object(raw_json, '$.session_key') AS INT) AS session_key,
        TRY_CAST(get_json_object(raw_json, '$.driver_number') AS INT) AS driver_number,
        get_json_object(raw_json, '$.full_name') AS full_name,
        get_json_object(raw_json, '$.name_acronym') AS name_acronym,
        get_json_object(raw_json, '$.team_name') AS team_name,
        get_json_object(raw_json, '$.team_colour') AS team_colour
    FROM
        workspace.formula_1.bronze_drivers
    WHERE
        get_json_object(raw_json, "$.error") IS NULL
  )
  SELECT DISTINCT
    session_key,
    driver_number,
    name_acronym,
    initcap(full_name) AS driver_name,
    team_name,
    CASE 
      WHEN team_colour IS NOT NULL THEN concat('#', team_colour) 
      ELSE '#FFFFFF' 
    END AS team_hex_colour
  FROM
    parsed_drivers
) AS source
ON target.session_key = source.session_key 
   AND target.driver_number = source.driver_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT driver_name) AS total_drivers,
    COUNT(DISTINCT team_name) AS total_teams 
FROM 
    workspace.formula_1.silver_dim_drivers

In [0]:
%sql
SELECT
*
FROM
workspace.formula_1.silver_dim_drivers
LIMIT 5